# Local PixelRAG: Chat With Your Own PDFs / Reports

This notebook builds a **local, private** PixelRAG-style pipeline for your own PDF files and reports .

Unlike the hosted `pixelrag.ai` API (which only serves  pre-built Wikipedia index), this notebook does everything locally:

```
Your PDFs  ->  Render pages to images  ->  Embed images (CLIP)  ->  Local vector index
                                                                          |
                                                                          v
Your Question  ->  Embed query (CLIP text encoder)  ->  Retrieve top-k page images
                                                                          |
                                                                          v
                                              Claude (vision model) reads the images -> Answer
```

No text extraction, no chunking of text, no OCR. Every page (table, chart, layout and all) is retrieved and read exactly as it looks.

**What you need before running this:**
- A folder of PDF files you want to search over
- An `ANTHROPIC_API_KEY` environment variable set (for the final "reading" step)
- ~5 minutes for the first run to download the CLIP model weights

## 1. Install dependencies

In [ ]:
%pip install --quiet pymupdf pillow numpy torch open_clip_torch anthropic tqdm

## 2. Configuration

Point `PDF_DIR` at the folder containing your PDFs. `TILE_DIR` and `INDEX_DIR` are working directories the notebook creates for you.

In [ ]:
import os
from pathlib import Path

PDF_DIR = Path("./my_pdfs")          # <-- put your PDFs here
TILE_DIR = Path("./pixel_tiles")     # rendered page images live here
INDEX_DIR = Path("./pixel_index")    # embeddings + metadata live here

RENDER_DPI = 150                     # higher = sharper but slower/more memory
MAX_TILE_HEIGHT_PX = 1600            # very tall pages get sliced into tiles of this height

TILE_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF source folder: {PDF_DIR.resolve()}")
print(f"Found {len(list(PDF_DIR.glob('*.pdf')))} PDF file(s).")

## 3. Render PDF pages to image tiles

This is the step that replaces text extraction/chunking entirely. Each page is rasterized with PyMuPDF. If a page renders taller than `MAX_TILE_HEIGHT_PX` (common for long scanned pages or continuous-form documents), it is sliced into fixed-height tiles — the same tiling idea PixelRAG uses for long web pages.

**Known limitation (inherited from PixelRAG itself):** fixed-height slicing can cut a table or paragraph in half if it straddles a tile boundary. For ordinary single-page-sized documents this rarely matters, since one PDF page almost always fits in one tile.

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import io

def render_pdf_to_tiles(pdf_path: Path, tile_dir: Path, dpi: int = RENDER_DPI,
                         max_tile_height: int = MAX_TILE_HEIGHT_PX):
    """Render every page of a PDF into one or more image tiles on disk.

    Returns a list of dicts: {tile_path, source_pdf, page_number, tile_index}
    """
    doc = fitz.open(pdf_path)
    zoom = dpi / 72  # PDF points are 72 per inch
    matrix = fitz.Matrix(zoom, zoom)
    records = []

    for page_number, page in enumerate(doc, start=1):
        pix = page.get_pixmap(matrix=matrix)
        img = Image.open(io.BytesIO(pix.tobytes("png")))

        width, height = img.size
        if height <= max_tile_height:
            tile_index = 0
            tile_path = tile_dir / f"{pdf_path.stem}_p{page_number:03d}_t{tile_index}.png"
            img.save(tile_path)
            records.append({
                "tile_path": str(tile_path),
                "source_pdf": pdf_path.name,
                "page_number": page_number,
                "tile_index": tile_index,
            })
        else:
            # slice a tall page into fixed-height tiles, like PixelRAG does for web pages
            n_tiles = (height + max_tile_height - 1) // max_tile_height
            for tile_index in range(n_tiles):
                top = tile_index * max_tile_height
                bottom = min(top + max_tile_height, height)
                crop = img.crop((0, top, width, bottom))
                tile_path = tile_dir / f"{pdf_path.stem}_p{page_number:03d}_t{tile_index}.png"
                crop.save(tile_path)
                records.append({
                    "tile_path": str(tile_path),
                    "source_pdf": pdf_path.name,
                    "page_number": page_number,
                    "tile_index": tile_index,
                })

    doc.close()
    return records

In [ ]:
from tqdm.auto import tqdm

all_tile_records = []
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

for pdf_path in tqdm(pdf_files, desc="Rendering PDFs"):
    all_tile_records.extend(render_pdf_to_tiles(pdf_path, TILE_DIR))

print(f"Rendered {len(all_tile_records)} tile(s) from {len(pdf_files)} PDF(s).")

## 4. Build the visual index (CLIP embeddings)

We embed every tile with CLIP so that later a **text** query can be matched against **image** tiles in the same vector space. This is the local stand-in for PixelRAG's hosted vision-language embedding step.

In [ ]:
import torch
import open_clip
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(device).eval()

In [ ]:
@torch.no_grad()
def embed_image_tiles(tile_paths, batch_size: int = 16):
    """Embed a list of image file paths and return an (N, D) normalized numpy array."""
    all_embeddings = []
    for i in tqdm(range(0, len(tile_paths), batch_size), desc="Embedding tiles"):
        batch_paths = tile_paths[i:i + batch_size]
        images = [clip_preprocess(Image.open(p).convert("RGB")) for p in batch_paths]
        batch_tensor = torch.stack(images).to(device)
        features = clip_model.encode_image(batch_tensor)
        features = features / features.norm(dim=-1, keepdim=True)
        all_embeddings.append(features.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)


@torch.no_grad()
def embed_text_query(query: str):
    """Embed a text query into the same space as the image tiles."""
    tokens = clip_tokenizer([query]).to(device)
    features = clip_model.encode_text(tokens)
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()[0]

In [ ]:
import json

tile_paths = [rec["tile_path"] for rec in all_tile_records]
embeddings = embed_image_tiles(tile_paths)

np.save(INDEX_DIR / "embeddings.npy", embeddings)
with open(INDEX_DIR / "metadata.json", "w") as f:
    json.dump(all_tile_records, f, indent=2)

print(f"Saved index: {embeddings.shape[0]} vectors of dimension {embeddings.shape[1]}")

## 5. Retrieval

Cosine similarity search over the local index. For a few thousand tiles, brute-force numpy is fast enough and needs no extra vector-database dependency. Swap in FAISS if your document collection grows into the tens of thousands of pages.

In [ ]:
def load_index():
    embeddings = np.load(INDEX_DIR / "embeddings.npy")
    with open(INDEX_DIR / "metadata.json") as f:
        metadata = json.load(f)
    return embeddings, metadata


def retrieve(query: str, k: int = 5):
    embeddings, metadata = load_index()
    query_vec = embed_text_query(query)
    scores = embeddings @ query_vec  # cosine similarity (vectors are already normalized)
    top_k_idx = np.argsort(-scores)[:k]
    return [
        {**metadata[i], "score": float(scores[i])}
        for i in top_k_idx
    ]

## 6. Reading step: hand the retrieved tiles to Claude

This is the step that makes PixelRAG different from a plain image search engine: the retrieved page images are sent as **images**, not as extracted text, to a vision-capable model, which reasons over tables, charts, and layout directly.

In [ ]:
import base64
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

def _image_to_base64_block(image_path: str):
    with open(image_path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return {
        "type": "image",
        "source": {"type": "base64", "media_type": "image/png", "data": data},
    }


def ask_vlm(query: str, retrieved_tiles: list):
    content_blocks = []
    for tile in retrieved_tiles:
        content_blocks.append(_image_to_base64_block(tile["tile_path"]))
        content_blocks.append({
            "type": "text",
            "text": f"(Source: {tile['source_pdf']}, page {tile['page_number']})",
        })

    content_blocks.append({
        "type": "text",
        "text": (
            f"Using only the page images above, answer this question: {query}\n\n"
            "Cite which source file and page number(s) you used. "
            "If the images don't contain the answer, say so directly."
        ),
    })

    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": content_blocks}],
    )
    return "".join(block.text for block in response.content if block.type == "text")

## 7. End-to-end pipeline

In [ ]:
def pixel_rag_query(query: str, k: int = 5, verbose: bool = True):
    retrieved = retrieve(query, k=k)
    if verbose:
        print(f"Retrieved {len(retrieved)} tile(s):")
        for r in retrieved:
            print(f"  - {r['source_pdf']} (page {r['page_number']}, tile {r['tile_index']}) score={r['score']:.3f}")
        print()
    answer = ask_vlm(query, retrieved)
    return answer, retrieved

## 8. Try it

Drop a few PDFs into `./my_pdfs`, re-run the rendering + indexing cells above, then ask a question. The example below shows the pattern; replace the question with something relevant to your own documents.

In [ ]:
question = "What was the total revenue reported in the Q3 table, and which quarter had the highest margin?"

answer, retrieved_tiles = pixel_rag_query(question, k=5)
print("ANSWER:\n")
print(answer)

In [ ]:
# Optional: visually inspect exactly which page images were retrieved
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(retrieved_tiles), figsize=(4 * len(retrieved_tiles), 6))
if len(retrieved_tiles) == 1:
    axes = [axes]
for ax, tile in zip(axes, retrieved_tiles):
    img = Image.open(tile["tile_path"])
    ax.imshow(img)
    ax.set_title(f"{tile['source_pdf']}\npage {tile['page_number']} ({tile['score']:.2f})", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Notes, costs, and next steps

- **Re-indexing:** re-run section 3 and 4 whenever you add or change PDFs in `my_pdfs/`. This notebook does a full rebuild each time for simplicity; for large, frequently changing collections, only render/embed new or modified files.
- **Scaling the index:** brute-force numpy search is fine up to a few thousand tiles. Beyond that, swap `retrieve()` to use FAISS (`faiss-cpu`) for approximate nearest-neighbor search.
- **Cost:** each query sends `k` full-resolution page images to Claude, at roughly ~1,500-1,600 tokens per image at 150 DPI. Lower `k` or `RENDER_DPI` to cut cost; raise them if answers are missing fine print or small chart labels.
- **Known limitation (inherited from PixelRAG):** fixed-height tile slicing has no awareness of content boundaries. If you regularly see a table or paragraph cut across a tile boundary, lower `MAX_TILE_HEIGHT_PX` won't help — increasing it (or keeping documents to one tile per page, as most normal PDFs already are) is the more reliable fix.
- **Privacy:** everything up through retrieval runs entirely on your machine. Only the final retrieved images are sent to the Claude API for the reading step.